# Section 02: Exploring Data Transformation with Google Cloud

**Companion notebook for**: `02-data-transformation.html`

## Learning Objectives
- Query data in BigQuery using the Python client
- Work with Cloud Storage buckets and objects
- Understand Pub/Sub messaging patterns
- Explore database service selection
- Build a data pipeline architecture

## Prerequisites
- A Google Cloud project with billing enabled (optional — mock outputs provided)
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-bigquery google-cloud-storage google-cloud-pubsub db-dtypes pandas

In [ ]:
import os
import json

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
print(f"Project ID: {PROJECT_ID}")

if PROJECT_ID == "your-project-id":
    print("WARNING: Mock outputs will be used. Set GCP_PROJECT_ID for live API calls.")

---
## Section 1: BigQuery — Querying Public Datasets

BigQuery is Google Cloud's serverless data warehouse. It can scan
terabytes of data in seconds using standard SQL. Public datasets
are available for free exploration.

In [ ]:
# Query a BigQuery public dataset
try:
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    
    query = """
    SELECT
        name,
        SUM(number) as total_count,
        MIN(year) as first_year,
        MAX(year) as last_year
    FROM `bigquery-public-data.usa_names.usa_1910_current`
    WHERE year >= 2000
    GROUP BY name
    ORDER BY total_count DESC
    LIMIT 10
    """
    
    df = client.query(query).to_dataframe()
    print("Top 10 baby names in the US (2000-present):\n")
    print(df.to_string(index=False))
    
    # Show query metadata
    job = client.query(query)
    job.result()
    print(f"\nData scanned: {job.total_bytes_processed / 1e6:.1f} MB")
    print(f"Query time:   {job.ended - job.started}")

except Exception as e:
    print(f"BigQuery call failed: {e}")
    print("\nMock output — Top baby names (2000-present):")
    mock_data = [
        ("Jacob", 273844, 2000, 2023),
        ("Emma", 264212, 2000, 2023),
        ("Michael", 253701, 2000, 2023),
        ("Olivia", 248638, 2000, 2023),
        ("Ethan", 239897, 2000, 2023),
        ("Isabella", 223472, 2000, 2023),
        ("Joshua", 219836, 2000, 2023),
        ("Sophia", 215388, 2000, 2023),
        ("Matthew", 213587, 2000, 2023),
        ("Liam", 209981, 2000, 2023),
    ]
    print(f"\n{'Name':<12} {'Count':>10} {'First':>6} {'Last':>6}")
    print("-" * 38)
    for name, count, first, last in mock_data:
        print(f"{name:<12} {count:>10,} {first:>6} {last:>6}")

---
## Section 2: BigQuery Cost Optimization

BigQuery on-demand pricing charges per TB scanned. Understanding
how to reduce data scanned is key for cost optimization.

In [ ]:
# BigQuery cost optimization techniques

optimization_tips = [
    {
        "technique": "Avoid SELECT *",
        "bad": "SELECT * FROM my_table",
        "good": "SELECT col1, col2 FROM my_table",
        "savings": "Often 80-95% less data scanned",
    },
    {
        "technique": "Use partitioned tables",
        "bad": "SELECT * FROM sales WHERE date > '2024-01-01'",
        "good": "SELECT * FROM sales WHERE _PARTITIONDATE > '2024-01-01'",
        "savings": "Scans only relevant date partitions",
    },
    {
        "technique": "Use clustered tables",
        "bad": "CREATE TABLE sales (...)",
        "good": "CREATE TABLE sales (...) CLUSTER BY customer_id, region",
        "savings": "Reduces data scanned for filtered queries",
    },
    {
        "technique": "Preview with LIMIT + dry run",
        "bad": "Run full query to see results",
        "good": "Use --dry_run flag or LIMIT clause for previewing",
        "savings": "Zero cost for dry runs",
    },
    {
        "technique": "Use materialized views",
        "bad": "Re-run expensive aggregation query repeatedly",
        "good": "CREATE MATERIALIZED VIEW ... AS SELECT ... GROUP BY ...",
        "savings": "Pre-computed results, incremental updates",
    },
]

print("BigQuery Cost Optimization Techniques")
print("=" * 60)
for tip in optimization_tips:
    print(f"\nTechnique: {tip['technique']}")
    print(f"  Bad:     {tip['bad']}")
    print(f"  Good:    {tip['good']}")
    print(f"  Savings: {tip['savings']}")

---
## Section 3: Cloud Storage — Buckets and Objects

Cloud Storage is object storage with 11 nines durability and four
storage classes for cost optimization.

In [ ]:
# Cloud Storage operations
try:
    from google.cloud import storage
    client = storage.Client(project=PROJECT_ID)
    
    # List buckets
    buckets = list(client.list_buckets())
    print(f"Buckets in project {PROJECT_ID}:")
    for b in buckets[:10]:
        print(f"  gs://{b.name}  (class: {b.storage_class}, location: {b.location})")

except Exception as e:
    print(f"Storage API failed: {e}")
    print("\nMock output — Cloud Storage example:")
    print("""    
    # Creating and using Cloud Storage with gsutil
    gsutil mb -l us-central1 gs://my-cdl-bucket
    gsutil cp local-file.csv gs://my-cdl-bucket/data/
    gsutil ls gs://my-cdl-bucket/data/
    gsutil cat gs://my-cdl-bucket/data/local-file.csv | head -5
    """)

In [ ]:
# Cloud Storage classes comparison
storage_classes = [
    ("Standard", "None", "~$0.020", "$0.00", "Frequently accessed, hot data"),
    ("Nearline", "30 days", "~$0.010", "$0.01/GB", "Accessed < once/month"),
    ("Coldline", "90 days", "~$0.004", "$0.02/GB", "Accessed < once/quarter"),
    ("Archive", "365 days", "~$0.0012", "$0.05/GB", "Accessed < once/year"),
]

print("Cloud Storage Classes")
print("=" * 85)
print(f"{'Class':<12} {'Min Duration':<15} {'Storage/GB/mo':<14} {'Retrieval/GB':<14} {'Use Case'}")
print("-" * 85)
for cls, dur, cost, ret, use in storage_classes:
    print(f"{cls:<12} {dur:<15} {cost:<14} {ret:<14} {use}")

print("\nAll classes have identical: durability (11 nines), latency (ms), and throughput.")
print("Difference is COST (storage vs retrieval tradeoff).")

---
## Section 4: Database Service Selection

Google Cloud offers multiple database services. Choosing the right one
depends on data type, access patterns, scale, and consistency requirements.

In [ ]:
# Database decision matrix
db_matrix = [
    {
        "scenario": "Web app with relational data, single region",
        "answer": "Cloud SQL",
        "why": "Standard SQL, managed MySQL/PostgreSQL, simple setup",
    },
    {
        "scenario": "Global financial system needing strong consistency",
        "answer": "Cloud Spanner",
        "why": "Globally distributed, strongly consistent, relational",
    },
    {
        "scenario": "Mobile app with offline sync and real-time updates",
        "answer": "Firestore",
        "why": "NoSQL document DB, real-time listeners, offline support",
    },
    {
        "scenario": "IoT sensor data, 1M+ writes/sec, time-series",
        "answer": "Cloud Bigtable",
        "why": "NoSQL wide-column, petabyte scale, sub-10ms latency",
    },
    {
        "scenario": "Caching layer for a web application",
        "answer": "Memorystore (Redis)",
        "why": "In-memory, sub-millisecond latency, managed Redis",
    },
    {
        "scenario": "Petabyte-scale analytics with SQL",
        "answer": "BigQuery",
        "why": "Serverless warehouse, not a transactional DB",
    },
    {
        "scenario": "Store images, videos, backups",
        "answer": "Cloud Storage",
        "why": "Object storage, not a database, but stores unstructured data",
    },
]

print("Database Service Decision Matrix")
print("=" * 80)
for item in db_matrix:
    print(f"\nScenario: {item['scenario']}")
    print(f"  Answer:  {item['answer']}")
    print(f"  Why:     {item['why']}")

---
## Section 5: Pub/Sub Messaging

Pub/Sub is the ingestion and messaging backbone for real-time data pipelines.
It decouples producers from consumers.

In [ ]:
# Pub/Sub concepts and patterns
try:
    from google.cloud import pubsub_v1
    publisher = pubsub_v1.PublisherClient()
    
    # List existing topics
    project_path = f"projects/{PROJECT_ID}"
    topics = list(publisher.list_topics(request={"project": project_path}))
    print(f"Topics in {PROJECT_ID}:")
    for t in topics:
        print(f"  {t.name}")
        
except Exception as e:
    print(f"Pub/Sub API failed: {e}")
    print("\nPub/Sub Architecture Pattern:")
    print("""
    Common real-time pipeline:
    
    [IoT Devices / Apps / Logs]
           |
    [ Cloud Pub/Sub Topic ]
           |
    [ Cloud Dataflow ]  <-- transforms, enriches, windows
           |
    [ BigQuery ]  <-- analytics and dashboards
           |
    [ Looker / Looker Studio ]  <-- visualization
    
    Key Pub/Sub concepts:
    - Topic: named channel for messages
    - Subscription: pull or push delivery to consumers
    - At-least-once delivery (may see duplicates)
    - Messages retained up to 31 days
    - Serverless, scales to millions of messages/sec
    """)

---
## Section 6: Dataflow vs Dataproc

Two data processing services with different strengths.

In [ ]:
# Dataflow vs Dataproc comparison
comparison = [
    ("Based on", "Apache Beam (Google-created)", "Apache Hadoop / Spark"),
    ("Management", "Fully serverless", "Managed clusters"),
    ("Batch + Stream", "Unified model for both", "Separate (Spark Streaming)"),
    ("Best for", "New pipelines, real-time ETL", "Existing Hadoop/Spark code"),
    ("Scaling", "Auto (no cluster config)", "You set cluster size"),
    ("Startup time", "Seconds", "~90 seconds"),
    ("Exam keyword", "'serverless data processing'", "'migrate existing Spark/Hadoop'"),
]

print("Dataflow vs Dataproc")
print("=" * 75)
print(f"{'Feature':<20} {'Dataflow':<28} {'Dataproc'}")
print("-" * 75)
for feature, df, dp in comparison:
    print(f"{feature:<20} {df:<28} {dp}")

print("\nDecision rule:")
print("  New pipeline with no existing code? -> Dataflow")
print("  Migrating existing Hadoop/Spark jobs? -> Dataproc")

---
## Summary

In this notebook we covered:

1. **BigQuery** — Serverless warehouse, SQL queries, public datasets, cost optimization.
2. **Cloud Storage** — Object storage, 4 classes (Standard/Nearline/Coldline/Archive), lifecycle rules.
3. **Database Selection** — Cloud SQL, Spanner, Firestore, Bigtable, Memorystore for different needs.
4. **Pub/Sub** — Messaging service for real-time pipelines, at-least-once delivery.
5. **Dataflow vs Dataproc** — Serverless Beam (new) vs managed Spark/Hadoop (migration).

**Next notebook**: Section 03 covers AI and ML — pre-trained APIs, AutoML, BigQuery ML, and Vertex AI.